# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; przygotowuje konfiguracje, modele, trening, ewaluację oraz zapis wyników.

## Konfiguracja

Importy, stałe projektu oraz podstawowe etykiety zadania.

In [ ]:
from dataclasses import dataclass, field, fields
from itertools import product
from pathlib import Path
from typing import Any, Literal
import json
import math
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchaudio

DATA_DIR = Path("data")
TRAIN_ARCHIVE = DATA_DIR / "train.7z"

TARGET_LABEL_ORDER = ("yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go")
UNKNOWN_LABEL = "unknown"
SILENCE_LABEL = "silence"
LABEL_ORDER = TARGET_LABEL_ORDER + (UNKNOWN_LABEL, SILENCE_LABEL)
LABEL_TO_ID = {label: index for index, label in enumerate(LABEL_ORDER)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}

## Interfejs eksperymentów

Dataclassy opisują dane, reprezentację audio, modele oraz trening. Pola w klasach `GridParams` mogą być pojedynczą wartością albo listą wartości.

In [ ]:
ModelType = Literal["lstm", "transformer"]
SamplingStrategy = Literal["natural", "class_balanced"]
Device = Literal["auto", "cpu", "cuda"]


@dataclass(frozen=True)
class DataFixedParams:
    data_dir: str = "data"
    train_archive: str = "train.7z"
    cache_dir: str = ".cache/baseline_audio"
    output_dir: str = "reports/02_baseline_models"
    target_labels: tuple[str, ...] = TARGET_LABEL_ORDER
    unknown_label: str = UNKNOWN_LABEL
    silence_label: str = SILENCE_LABEL
    sample_rate: int = 16_000
    clip_seconds: float = 1.0
    include_silence: bool = True


@dataclass(frozen=True)
class DataGridParams:
    train_fraction: float | list[float] = 1.0
    test_fraction: float | list[float] = 1.0
    unknown_fraction: float | list[float] = 1.0
    silence_examples_per_split: int | list[int] = 2_000
    sampling_strategy: SamplingStrategy | list[SamplingStrategy] = "natural"
    seed: int | list[int] = 42


@dataclass(frozen=True)
class FeatureFixedParams:
    n_mels: int = 64
    n_fft: int = 512
    hop_length: int = 160
    normalize: bool = True


@dataclass(frozen=True)
class ModelGridParams:
    model_type: ModelType | list[ModelType] = field(default_factory=lambda: ["lstm", "transformer"])
    dropout: float | list[float] = 0.2

    lstm_hidden_size: int | list[int] = 128
    lstm_layers: int | list[int] = 2
    lstm_bidirectional: bool | list[bool] = True

    transformer_d_model: int | list[int] = 128
    transformer_heads: int | list[int] = 4
    transformer_layers: int | list[int] = 2
    transformer_ff_dim: int | list[int] = 256


@dataclass(frozen=True)
class FitFixedParams:
    device: Device = "auto"
    num_workers: int = 0
    pin_memory: bool = False
    log_every: int = 20


@dataclass(frozen=True)
class FitGridParams:
    epochs: int | list[int] = 5
    batch_size: int | list[int] = 64
    learning_rate: float | list[float] = 1e-3
    weight_decay: float | list[float] = 1e-4


@dataclass(frozen=True)
class Experiment:
    name: str
    data_fixed: DataFixedParams = DataFixedParams()
    data_grid: DataGridParams = DataGridParams()
    feature_fixed: FeatureFixedParams = FeatureFixedParams()
    model_grid: ModelGridParams = ModelGridParams()
    fit_fixed: FitFixedParams = FitFixedParams()
    fit_grid: FitGridParams = FitGridParams()

In [ ]:
def as_list(value: Any) -> list[Any]:
    return value if isinstance(value, list) else [value]


def expand_grid(dataclass_instance: Any) -> list[dict[str, Any]]:
    keys = [field.name for field in fields(dataclass_instance)]
    values = [as_list(getattr(dataclass_instance, key)) for key in keys]
    return [dict(zip(keys, combination)) for combination in product(*values)]


def expand_experiment_grid(experiment: Experiment) -> list[dict[str, Any]]:
    runs = []
    for data_params, model_params, fit_params in product(
        expand_grid(experiment.data_grid),
        expand_grid(experiment.model_grid),
        expand_grid(experiment.fit_grid),
    ):
        runs.append(
            {
                "experiment": experiment.name,
                "data": data_params,
                "model": model_params,
                "fit": fit_params,
            }
        )
    return runs


def experiment_grid_dataframe(experiment: Experiment) -> pd.DataFrame:
    rows = []
    for run in expand_experiment_grid(experiment):
        rows.append(
            {
                "experiment": run["experiment"],
                **{f"data.{key}": value for key, value in run["data"].items()},
                **{f"model.{key}": value for key, value in run["model"].items()},
                **{f"fit.{key}": value for key, value in run["fit"].items()},
            }
        )
    return pd.DataFrame(rows)

## Manifest danych

Kod buduje manifest etykietowanych nagrań z `train.7z`, korzystając z oficjalnych list `validation_list.txt` i `testing_list.txt`. Kaggle `test.7z` nie jest używany do metryk, bo nie zawiera etykiet.

In [ ]:
def archive_files(archive: Path) -> list[Path]:
    result = subprocess.run(
        ["7z", "l", "-slt", str(archive)],
        check=True,
        capture_output=True,
        text=True,
    )

    files = []
    entry = {}

    def add_file() -> None:
        if entry.get("Path") and "Size" in entry and not entry.get("Attributes", "").startswith("D"):
            files.append(Path(entry["Path"]))

    for line in result.stdout.splitlines():
        if not line:
            add_file()
            entry = {}
            continue

        if " = " in line:
            key, value = line.split(" = ", 1)
            entry[key] = value

    add_file()
    return files


def archive_lines(archive: Path, file_name: str) -> set[Path]:
    result = subprocess.run(
        ["7z", "e", "-so", str(archive), file_name],
        check=True,
        capture_output=True,
        text=True,
    )
    return {Path(line) for line in result.stdout.splitlines() if line}


def train_audio_label(path: Path) -> str | None:
    parts = path.parts
    if len(parts) != 4 or parts[:2] != ("train", "audio") or path.suffix != ".wav":
        return None
    return parts[2]


def speaker_id(path: Path) -> str:
    return path.name.split("_nohash_", 1)[0]

In [ ]:
def build_command_manifest(data_params: DataFixedParams) -> pd.DataFrame:
    archive = Path(data_params.data_dir) / data_params.train_archive
    validation_paths = archive_lines(archive, "train/validation_list.txt")
    testing_paths = archive_lines(archive, "train/testing_list.txt")

    rows = []
    for archive_path in archive_files(archive):
        label = train_audio_label(archive_path)
        if label is None or label == "_background_noise_":
            continue

        relative_path = Path(*archive_path.parts[2:])
        split = "test" if relative_path in testing_paths else "validation" if relative_path in validation_paths else "train"
        mapped_label = label if label in data_params.target_labels else data_params.unknown_label

        rows.append(
            {
                "archive_path": str(archive_path),
                "relative_path": str(relative_path),
                "original_label": label,
                "label": mapped_label,
                "split": split,
                "speaker_id": speaker_id(archive_path),
                "segment_index": 0,
            }
        )

    return pd.DataFrame(rows)


def background_noise_paths(data_params: DataFixedParams) -> list[Path]:
    archive = Path(data_params.data_dir) / data_params.train_archive
    return [
        path for path in archive_files(archive)
        if len(path.parts) == 4 and path.parts[:3] == ("train", "audio", "_background_noise_") and path.suffix == ".wav"
    ]


def add_silence_examples(manifest: pd.DataFrame, data_params: DataFixedParams, examples_per_split: int) -> pd.DataFrame:
    if not data_params.include_silence or examples_per_split <= 0:
        return manifest

    noise_paths = background_noise_paths(data_params)
    rows = []
    for split in ("train", "test"):
        for index in range(examples_per_split):
            path = noise_paths[index % len(noise_paths)]
            rows.append(
                {
                    "archive_path": str(path),
                    "relative_path": str(Path(*path.parts[2:])),
                    "original_label": "_background_noise_",
                    "label": data_params.silence_label,
                    "split": split,
                    "speaker_id": f"background_{index % len(noise_paths)}",
                    "segment_index": index,
                }
            )

    return pd.concat([manifest, pd.DataFrame(rows)], ignore_index=True)

In [ ]:
def sample_split(manifest: pd.DataFrame, split: str, fraction: float, unknown_fraction: float, seed: int) -> pd.DataFrame:
    split_data = manifest[manifest["split"] == split].copy()
    sampled = []

    for label, group in split_data.groupby("label", sort=False):
        label_fraction = unknown_fraction if label == UNKNOWN_LABEL else fraction
        if label_fraction >= 1.0:
            sampled.append(group)
            continue

        count = max(1, math.ceil(len(group) * label_fraction))
        sampled.append(group.sample(n=count, random_state=seed))

    return pd.concat(sampled, ignore_index=True)


def build_experiment_manifest(
    data_params: DataFixedParams,
    train_fraction: float,
    test_fraction: float,
    unknown_fraction: float,
    silence_examples_per_split: int,
    seed: int,
) -> pd.DataFrame:
    manifest = build_command_manifest(data_params)
    manifest = add_silence_examples(manifest, data_params, silence_examples_per_split)

    train_manifest = sample_split(manifest, "train", train_fraction, unknown_fraction, seed)
    test_manifest = sample_split(manifest, "test", test_fraction, unknown_fraction, seed)
    return pd.concat([train_manifest, test_manifest], ignore_index=True)

## Dataset i cechy audio

Dataset ładuje wybrane pliki WAV, resampluje je do jednej częstotliwości, przycina lub dopełnia do stałej długości i zwraca mel-spektrogram w układzie sekwencyjnym `[czas, pasma_mel]`.

In [ ]:
def extract_archive_files(archive: Path, paths: list[Path], output_dir: Path) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    unique_paths = list(dict.fromkeys(paths))

    if len(unique_paths) > 5_000:
        command = ["7z", "x", "-y", str(archive), f"-o{output_dir}", "train/audio"]
    else:
        command = ["7z", "x", "-y", str(archive), f"-o{output_dir}"] + [str(path) for path in unique_paths]

    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return {str(path): output_dir / path for path in unique_paths}


def pad_or_trim(waveform: torch.Tensor, target_length: int, segment_index: int = 0) -> torch.Tensor:
    if waveform.numel() < target_length:
        return torch.nn.functional.pad(waveform, (0, target_length - waveform.numel()))

    if waveform.numel() == target_length:
        return waveform

    max_start = waveform.numel() - target_length
    start = 0 if segment_index == 0 else (segment_index * target_length) % (max_start + 1)
    return waveform[start : start + target_length]

In [ ]:
class SpeechCommandsDataset(Dataset):
    def __init__(
        self,
        manifest: pd.DataFrame,
        local_paths: dict[str, Path],
        data_params: DataFixedParams,
        feature_params: FeatureFixedParams,
    ):
        self.manifest = manifest.reset_index(drop=True)
        self.local_paths = local_paths
        self.data_params = data_params
        self.feature_params = feature_params
        self.target_length = int(data_params.sample_rate * data_params.clip_seconds)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=data_params.sample_rate,
            n_fft=feature_params.n_fft,
            hop_length=feature_params.hop_length,
            n_mels=feature_params.n_mels,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.manifest.iloc[index]
        waveform, sample_rate = torchaudio.load(self.local_paths[row.archive_path])
        waveform = waveform.mean(dim=0)

        if sample_rate != self.data_params.sample_rate:
            waveform = torchaudio.functional.resample(waveform, sample_rate, self.data_params.sample_rate)

        waveform = pad_or_trim(waveform, self.target_length, int(row.segment_index))
        features = self.to_db(self.mel(waveform)).transpose(0, 1)

        if self.feature_params.normalize:
            features = (features - features.mean()) / (features.std().clamp_min(1e-6))

        return features, torch.tensor(LABEL_TO_ID[row.label], dtype=torch.long)

In [ ]:
def make_dataloaders(
    experiment: Experiment,
    data_grid: dict[str, Any],
    fit_grid: dict[str, Any],
) -> tuple[DataLoader, DataLoader, pd.DataFrame]:
    data_params = experiment.data_fixed
    manifest = build_experiment_manifest(
        data_params=data_params,
        train_fraction=data_grid["train_fraction"],
        test_fraction=data_grid["test_fraction"],
        unknown_fraction=data_grid["unknown_fraction"],
        silence_examples_per_split=data_grid["silence_examples_per_split"],
        seed=data_grid["seed"],
    )

    archive = Path(data_params.data_dir) / data_params.train_archive
    cache_dir = Path(data_params.cache_dir) / experiment.name
    local_paths = extract_archive_files(archive, [Path(path) for path in manifest["archive_path"]], cache_dir)

    train_manifest = manifest[manifest["split"] == "train"]
    test_manifest = manifest[manifest["split"] == "test"]
    train_dataset = SpeechCommandsDataset(train_manifest, local_paths, data_params, experiment.feature_fixed)
    test_dataset = SpeechCommandsDataset(test_manifest, local_paths, data_params, experiment.feature_fixed)

    sampler = None
    shuffle = True
    if data_grid["sampling_strategy"] == "class_balanced":
        label_counts = train_manifest["label"].value_counts().to_dict()
        weights = [1.0 / label_counts[label] for label in train_manifest["label"]]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        shuffle = False

    train_loader = DataLoader(
        train_dataset,
        batch_size=fit_grid["batch_size"],
        shuffle=shuffle,
        sampler=sampler,
        num_workers=experiment.fit_fixed.num_workers,
        pin_memory=experiment.fit_fixed.pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=fit_grid["batch_size"],
        shuffle=False,
        num_workers=experiment.fit_fixed.num_workers,
        pin_memory=experiment.fit_fixed.pin_memory,
    )

    return train_loader, test_loader, manifest

## Baseline modele

Dwa proste modele korzystają z tej samej reprezentacji wejściowej. LSTM przetwarza spektrogram sekwencyjnie, a Transformer używa samouwagi po projekcji pasm mel do wymiaru modelu.

In [ ]:
class LSTMBaseline(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, bidirectional: bool, dropout: float, num_classes: int):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        direction_multiplier = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * direction_multiplier, num_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        output, _ = self.lstm(inputs)
        pooled = output.mean(dim=1)
        return self.classifier(pooled)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10_000.0) / d_model))
        encoding = torch.zeros(max_len, d_model)
        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term[: encoding[:, 1::2].shape[1]])
        self.register_buffer("encoding", encoding.unsqueeze(0))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return inputs + self.encoding[:, : inputs.size(1)]


class TransformerBaseline(nn.Module):
    def __init__(self, input_size: int, d_model: int, heads: int, layers: int, ff_dim: int, dropout: float, num_classes: int):
        super().__init__()
        self.projection = nn.Linear(input_size, d_model)
        self.position = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_classes))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(self.position(self.projection(inputs)))
        pooled = encoded.mean(dim=1)
        return self.classifier(pooled)

In [ ]:
def build_model(model_params: dict[str, Any], feature_params: FeatureFixedParams, num_classes: int) -> nn.Module:
    if model_params["model_type"] == "lstm":
        return LSTMBaseline(
            input_size=feature_params.n_mels,
            hidden_size=model_params["lstm_hidden_size"],
            num_layers=model_params["lstm_layers"],
            bidirectional=model_params["lstm_bidirectional"],
            dropout=model_params["dropout"],
            num_classes=num_classes,
        )

    return TransformerBaseline(
        input_size=feature_params.n_mels,
        d_model=model_params["transformer_d_model"],
        heads=model_params["transformer_heads"],
        layers=model_params["transformer_layers"],
        ff_dim=model_params["transformer_ff_dim"],
        dropout=model_params["dropout"],
        num_classes=num_classes,
    )

## Trening i ewaluacja

Funkcje trenują pojedynczą konfigurację, liczą stratę, accuracy oraz macierz pomyłek. Notebook nie wywołuje ich automatycznie.

In [ ]:
def resolve_device(device: Device) -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device) -> dict[str, float]:
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return {"loss": total_loss / total, "accuracy": correct / total}


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> dict[str, Any]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    predictions = []
    targets = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            predicted = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            predictions.extend(predicted.cpu().tolist())
            targets.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "predictions": predictions,
        "targets": targets,
    }


def fit_model(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, fit_params: dict[str, Any], fixed_params: FitFixedParams) -> tuple[list[dict[str, float]], dict[str, Any]]:
    device = resolve_device(fixed_params.device)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=fit_params["learning_rate"], weight_decay=fit_params["weight_decay"])

    history = []
    for epoch in range(1, fit_params["epochs"] + 1):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_metrics = evaluate(model, test_loader, criterion, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
            }
        )

    final_metrics = evaluate(model, test_loader, criterion, device)
    return history, final_metrics

In [ ]:
def confusion_matrix(targets: list[int], predictions: list[int], num_classes: int) -> np.ndarray:
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    return matrix


def metrics_summary(history: list[dict[str, float]], final_metrics: dict[str, Any]) -> dict[str, float]:
    last = history[-1]
    return {
        "train_loss": last["train_loss"],
        "train_accuracy": last["train_accuracy"],
        "test_loss": final_metrics["loss"],
        "test_accuracy": final_metrics["accuracy"],
    }

## Zapisywanie wyników

Każde uruchomienie zapisuje konfigurację, metryki i wykresy w osobnym katalogu `reports/02_baseline_models/<experiment>/<run>/`.

In [ ]:
def slugify(value: str) -> str:
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def run_name(run: dict[str, Any]) -> str:
    model = run["model"]
    data = run["data"]
    fit = run["fit"]
    return slugify(
        f"{model['model_type']}_train{data['train_fraction']}_test{data['test_fraction']}_lr{fit['learning_rate']}_seed{data['seed']}"
    )


def output_paths(experiment: Experiment, run: dict[str, Any]) -> dict[str, Path]:
    root = Path(experiment.data_fixed.output_dir) / experiment.name / run_name(run)
    paths = {
        "root": root,
        "figures": root / "figures",
        "metrics": root / "metrics",
        "configs": root / "configs",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(path: Path, data: dict[str, Any]) -> None:
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))


def save_history_plot(history: list[dict[str, float]], path: Path) -> None:
    data = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(data["epoch"], data["train_loss"], label="train")
    axes[0].plot(data["epoch"], data["test_loss"], label="test")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(data["epoch"], data["train_accuracy"], label="train")
    axes[1].plot(data["epoch"], data["test_accuracy"], label="test")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.savefig(path, dpi=150)
    plt.close(fig)


def save_confusion_matrix_plot(matrix: np.ndarray, labels: tuple[str, ...], path: Path) -> None:
    fig, axis = plt.subplots(figsize=(9, 8), constrained_layout=True)
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_title("Confusion Matrix")
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
    axis.set_xticks(range(len(labels)), labels=labels, rotation=45, ha="right")
    axis.set_yticks(range(len(labels)), labels=labels)
    fig.colorbar(image, ax=axis, shrink=0.8)
    fig.savefig(path, dpi=150)
    plt.close(fig)

## Runner eksperymentów

`run_experiment` przechodzi po siatce konfiguracji, trenuje model, zapisuje wykresy i zwraca tabelę wyników. Komórka z wywołaniem jest celowo zakomentowana.

In [ ]:
def run_single_config(experiment: Experiment, run: dict[str, Any]) -> dict[str, Any]:
    torch.manual_seed(run["data"]["seed"])
    np.random.seed(run["data"]["seed"])

    paths = output_paths(experiment, run)
    save_json(paths["configs"] / "config.json", {"experiment": experiment.name, **run})

    train_loader, test_loader, manifest = make_dataloaders(experiment, run["data"], run["fit"])
    manifest.to_csv(paths["metrics"] / "manifest.csv", index=False)

    model = build_model(run["model"], experiment.feature_fixed, len(LABEL_ORDER))
    history, final_metrics = fit_model(model, train_loader, test_loader, run["fit"], experiment.fit_fixed)

    summary = metrics_summary(history, final_metrics)
    pd.DataFrame(history).to_csv(paths["metrics"] / "history.csv", index=False)
    pd.DataFrame([summary]).to_csv(paths["metrics"] / "summary.csv", index=False)

    matrix = confusion_matrix(final_metrics["targets"], final_metrics["predictions"], len(LABEL_ORDER))
    pd.DataFrame(matrix, index=LABEL_ORDER, columns=LABEL_ORDER).to_csv(paths["metrics"] / "confusion_matrix.csv")

    save_history_plot(history, paths["figures"] / "learning_curves.png")
    save_confusion_matrix_plot(matrix, LABEL_ORDER, paths["figures"] / "confusion_matrix.png")

    return {"run": run_name(run), **summary}


def run_experiment(experiment: Experiment) -> pd.DataFrame:
    results = [run_single_config(experiment, run) for run in expand_experiment_grid(experiment)]
    summary = pd.DataFrame(results)
    output_dir = Path(experiment.data_fixed.output_dir) / experiment.name
    output_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(output_dir / "experiment_summary.csv", index=False)
    return summary

## Definicja baseline eksperymentu

Domyślne klasy konfiguracyjne używają pełnych danych. Poniższy eksperyment startowy jest ustawiony na 5% splitu treningowego i 5% splitu testowego, żeby można było szybko sprawdzić pipeline po odkomentowaniu uruchomienia.

In [ ]:
baseline_experiment = Experiment(
    name="baseline_lstm_transformer_5_percent",
    data_grid=DataGridParams(
        train_fraction=0.05,
        test_fraction=0.05,
        unknown_fraction=0.05,
        silence_examples_per_split=100,
        sampling_strategy="natural",
        seed=42,
    ),
    model_grid=ModelGridParams(
        model_type=["lstm", "transformer"],
        dropout=0.2,
        lstm_hidden_size=128,
        lstm_layers=2,
        lstm_bidirectional=True,
        transformer_d_model=128,
        transformer_heads=4,
        transformer_layers=2,
        transformer_ff_dim=256,
    ),
    fit_grid=FitGridParams(
        epochs=5,
        batch_size=64,
        learning_rate=1e-3,
        weight_decay=1e-4,
    ),
)

experiment_grid_dataframe(baseline_experiment)

In [ ]:
# Uruchomienie jest celowo wyłączone, żeby notebook nie przeprowadzał eksperymentów automatycznie.
# results = run_experiment(baseline_experiment)
# results